# Chest X-ray Model Training (Local GPU)

This notebook trains multiple transfer-learning backbones on the dataset in `archive/`.

Update `DATA_DIR` if your dataset lives elsewhere. Models are saved to `SAVE_DIR`.


In [1]:
import os
from pathlib import Path

import tensorflow as tf
from tensorflow import keras

print("TensorFlow:", tf.__version__)

# Enable GPU memory growth (safe for most local setups)
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as exc:
        print("Memory growth not set:", exc)

# Update these paths as needed
DATA_DIR = Path(r"c:\Users\Ayush\Desktop\My coding documents\AI\Computer Vision\Transfer Learning\archive")
SAVE_DIR = Path(r"c:\Users\Ayush\Desktop\My coding documents\AI\pneumo-tb-normal-classifier\models_trained")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

AUTOTUNE = tf.data.AUTOTUNE


TensorFlow: 2.20.0


In [2]:
def build_datasets(img_size, batch_size):
    train_ds = keras.utils.image_dataset_from_directory(
        DATA_DIR / "train",
        image_size=img_size,
        batch_size=batch_size,
        label_mode="categorical",
        shuffle=True,
    )
    val_ds = keras.utils.image_dataset_from_directory(
        DATA_DIR / "val",
        image_size=img_size,
        batch_size=batch_size,
        label_mode="categorical",
        shuffle=False,
    )
    test_ds = keras.utils.image_dataset_from_directory(
        DATA_DIR / "test",
        image_size=img_size,
        batch_size=batch_size,
        label_mode="categorical",
        shuffle=False,
    )
    class_names = train_ds.class_names
    return train_ds, val_ds, test_ds, class_names


def compute_class_weights():
    counts = {}
    for cls_dir in (DATA_DIR / "train").iterdir():
        if cls_dir.is_dir():
            counts[cls_dir.name] = len(list(cls_dir.glob("*.jpg")))
    total = sum(counts.values())
    class_names = sorted(counts.keys())
    class_weight = {
        i: total / (len(class_names) * counts[name])
        for i, name in enumerate(class_names)
    }
    return class_weight


class_weight = compute_class_weights()
print("class_weight:", class_weight)


class_weight: {0: 0.9547619047619048, 1: 0.9901234567901235, 2: 1.060846560846561}


In [3]:
from tensorflow.keras.applications import (
    DenseNet121,
    DenseNet169,
    ResNet50,
    ResNet50V2,
    EfficientNetB0,
    EfficientNetB2,
    EfficientNetB4,
    EfficientNetB5,
    InceptionResNetV2,
    MobileNetV2,
    MobileNetV3Large,
    NASNetMobile,
    EfficientNetV2B0,
    EfficientNetV2B1,
)

from tensorflow.keras.applications import (
    densenet,
    resnet,
    resnet_v2,
    efficientnet,
    inception_resnet_v2,
    mobilenet_v2,
    mobilenet_v3,
    nasnet,
    efficientnet_v2,
)

MODEL_ZOO = [
    ("DenseNet121", DenseNet121, densenet.preprocess_input, (224, 224), 32),
    ("DenseNet169", DenseNet169, densenet.preprocess_input, (224, 224), 24),
    ("ResNet50", ResNet50, resnet.preprocess_input, (224, 224), 32),
    ("ResNet50V2", ResNet50V2, resnet_v2.preprocess_input, (224, 224), 32),
    ("EfficientNetB0", EfficientNetB0, efficientnet.preprocess_input, (224, 224), 32),
    ("EfficientNetB2", EfficientNetB2, efficientnet.preprocess_input, (260, 260), 24),
    ("EfficientNetB4", EfficientNetB4, efficientnet.preprocess_input, (380, 380), 16),
    ("EfficientNetB5", EfficientNetB5, efficientnet.preprocess_input, (456, 456), 8),
    ("InceptionResNetV2", InceptionResNetV2, inception_resnet_v2.preprocess_input, (299, 299), 8),
    ("MobileNetV2", MobileNetV2, mobilenet_v2.preprocess_input, (224, 224), 64),
    ("MobileNetV3Large", MobileNetV3Large, mobilenet_v3.preprocess_input, (224, 224), 48),
    ("NASNetMobile", NASNetMobile, nasnet.preprocess_input, (224, 224), 32),
    ("EfficientNetV2B0", EfficientNetV2B0, efficientnet_v2.preprocess_input, (224, 224), 32),
    ("EfficientNetV2B1", EfficientNetV2B1, efficientnet_v2.preprocess_input, (240, 240), 24),
]

# Optionally restrict which models to run
# Example: ENABLED_MODELS = {"DenseNet121", "EfficientNetB0"}
ENABLED_MODELS = set()


In [4]:
def add_preprocess(ds, preprocess_fn):
    def _map(x, y):
        x = tf.cast(x, tf.float32)
        x = preprocess_fn(x)
        return x, y

    return ds.map(_map, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


def build_model(base_model_fn, input_shape, num_classes=3):
    base = base_model_fn(
        input_shape=input_shape,
        weights="imagenet",
        include_top=False,
    )
    base.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    return model, base


def train_one(model_name, base_fn, preprocess_fn, img_size, batch_size, class_weight):
    print(f"\n=== Training {model_name} ===")

    train_ds, val_ds, test_ds, class_names = build_datasets(img_size, batch_size)
    train_ds = add_preprocess(train_ds, preprocess_fn)
    val_ds = add_preprocess(val_ds, preprocess_fn)
    test_ds = add_preprocess(test_ds, preprocess_fn)

    model, base = build_model(base_fn, (*img_size, 3), num_classes=len(class_names))

    callbacks = [
        keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor="val_loss"),
        keras.callbacks.ModelCheckpoint(
            filepath=str(SAVE_DIR / f"{model_name}.keras"),
            monitor="val_loss",
            save_best_only=True,
        ),
    ]

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=8,
        class_weight=class_weight,
        callbacks=callbacks,
    )

    base.trainable = True
    for layer in base.layers[:-40]:
        layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(1e-5),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        class_weight=class_weight,
        callbacks=callbacks,
    )

    val_loss, val_acc = model.evaluate(val_ds, verbose=0)
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    print(f"{model_name} val_acc={val_acc:.4f} test_acc={test_acc:.4f}")


In [ ]:
def should_run(name):
    return not ENABLED_MODELS or name in ENABLED_MODELS


for name, base_fn, preprocess_fn, img_size, batch_size in MODEL_ZOO:
    if should_run(name):
        train_one(name, base_fn, preprocess_fn, img_size, batch_size, class_weight)
    else:
        print("Skipping", name)



=== Training DenseNet121 ===
Found 4812 files belonging to 3 classes.
Found 2534 files belonging to 3 classes.
Found 2569 files belonging to 3 classes.
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Epoch 1/8
151/151 ━━━━━━━━━━━━━━━━━━━━ 561s 4s/step - accuracy: 0.6473 - loss: 0.8011 - val_accuracy: 0.7427 - val_loss: 0.5383
Epoch 2/8
 22/151 ━━━━━━━━━━━━━━━━━━━━ 5:13 2s/step - accuracy: 0.7699 - loss: 0.5458

KeyboardInterrupt: 

: 